In [6]:
import numpy as np
import pandas as pd
import librosa
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

TensorFlow Version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [7]:
# Configuration
SAMPLE_RATE = 32000 # 32kHz as specified in competition
DURATION = 5 # 5 seconds per segment
N_MELS = 128 # mel spectogram bins
HOP_LENGTH = 512 # hop length for spectogram
N_FFT = 1024 # FFT window size
TARGET_SHAPE = (128, 128) # image size for CNN

# Paths
TRAIN_AUDIO = '/kaggle/input/competitions/birdclef-2026/train_audio/'
TRAIN_CSV = '/kaggle/input/competitions/birdclef-2026/train.csv'

# Load metadata
train = pd.read_csv(TRAIN_CSV)
print(f'Train shape: {train.shape}')
print(f"Species count: {train['primary_label'].nunique()}")

Train shape: (35549, 15)
Species count: 206


In [8]:
from sklearn.preprocessing import LabelEncoder

# Encode species labels
le = LabelEncoder()
train['label_encoded'] = le.fit_transform(train['primary_label'])

NUM_CLASSES = len(le.classes_)
print(f'Number of Classes: {NUM_CLASSES}')
print(f'\nSample Encoding')
print(train[['primary_label', 'label_encoded']].head(10))

Number of Classes: 206

Sample Encoding
  primary_label  label_encoded
0       1161364              0
1       1161364              0
2       1161364              0
3       1161364              0
4       1161364              0
5       1161364              0
6       1161364              0
7       1161364              0
8       1161364              0
9       1161364              0


In [9]:
def audio_to_melspectrogram(audio_path, duration=DURATION, sr=SAMPLE_RATE):
    try:
        y, sr = librosa.load(audio_path, sr=sr, duration=duration)
        
        target_length = sr * duration
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)))
        
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=N_MELS,
            hop_length=HOP_LENGTH, n_fft=N_FFT)
        
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # Fixed typo — mel_db not meldb
        mel_db = tf.image.resize(mel_db[..., np.newaxis], TARGET_SHAPE)
        mel_db = tf.squeeze(mel_db).numpy()
        
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        
        return mel_db
    
    except Exception as e:
        print(f"Error: {e}")
        return np.zeros(TARGET_SHAPE)

# Test again
sample = train['filename'].iloc[0]
full_path = TRAIN_AUDIO + sample
print(f"Full path: {full_path}")
print(f"File exists: {os.path.exists(full_path)}")

# Test function again
spec = audio_to_melspectrogram(full_path)
print(f"Shape: {spec.shape}")
print(f"Min: {spec.min():.3f}, Max: {spec.max():.3f}")

Full path: /kaggle/input/competitions/birdclef-2026/train_audio/1161364/iNat1216197.ogg
File exists: True
Shape: (128, 128)
Min: 0.000, Max: 1.000


I0000 00:00:1773311782.475491      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773311782.481805      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [10]:
# Use subset for faster training
# 20 samples per species × 206 species = 4120 samples
SAMPLES_PER_SPECIES = 20

train_subset = train.groupby('primary_label').apply(
    lambda x: x.sample(min(len(x), SAMPLES_PER_SPECIES), 
                       random_state=42)
).reset_index(drop=True)

print(f"Subset size: {len(train_subset)}")
print(f"Species count: {train_subset['primary_label'].nunique()}")

Subset size: 3652
Species count: 206


In [11]:
# Load all spectrograms
X = []
y_labels = []

print("Loading spectrograms...")
for idx, row in train_subset.iterrows():
    audio_path = TRAIN_AUDIO + row['filename']
    spec = audio_to_melspectrogram(audio_path)
    X.append(spec)
    y_labels.append(row['label_encoded'])
    
    if idx % 500 == 0:
        print(f"Processed {len(X)}/{len(train_subset)}")

X = np.array(X)
y_labels = np.array(y_labels)

print(f"\nDone!")
print(f"X shape: {X.shape}")
print(f"y shape: {y_labels.shape}")


Loading spectrograms...
Processed 1/3652
Processed 501/3652
Processed 1001/3652
Processed 1501/3652
Processed 2001/3652
Processed 2501/3652
Processed 3001/3652
Processed 3501/3652

Done!
X shape: (3652, 128, 128)
y shape: (3652,)


In [12]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Add channel dimension for CNN (needs 4D input)
X = X[..., np.newaxis]  # (3652, 128, 128, 1)

# One-hot encode labels
y_cat = to_categorical(y_labels, num_classes=NUM_CLASSES)

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_val:   {y_val.shape}")

X_train: (2921, 128, 128, 1)
X_val:   (731, 128, 128, 1)
y_train: (2921, 206)
y_val:   (731, 206)


In [13]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization

model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 1)),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    # Block 2
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    # Block 3
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    
    # Classifier
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 206)            │        52,942 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,569,294 (25.06 MB)

 Trainable params: 6,568,846 (25.06 MB)

 Non-trainable params: 448 (1.75 KB)

In [14]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True)
    ]
)

Epoch 1/20


I0000 00:00:1773311905.531110     149 service.cc:152] XLA service 0x7f0bc4002320 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773311905.531158     149 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1773311905.531163     149 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1773311906.115178     149 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-12 10:38:27.938179: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 10:38:28.086304: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


 5/92 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.0077 - loss: 12.3927

I0000 00:00:1773311911.391516     149 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0047 - loss: 8.1596

2026-03-12 10:38:34.485564: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-12 10:38:34.629318: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


92/92 ━━━━━━━━━━━━━━━━━━━━ 17s 94ms/step - accuracy: 0.0047 - loss: 8.1197 - val_accuracy: 0.0014 - val_loss: 34.5253
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0068 - loss: 5.3335 - val_accuracy: 0.0014 - val_loss: 40.8059
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0037 - loss: 5.3333 - val_accuracy: 0.0014 - val_loss: 36.8795
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0014 - loss: 5.3101 - val_accuracy: 0.0027 - val_loss: 31.5296
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0035 - loss: 5.3279 - val_accuracy: 0.0082 - val_loss: 12.3748
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0092 - loss: 5.3053 - val_accuracy: 0.0055 - val_loss: 5.9363
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0063 - loss: 5.3296 - val_accuracy: 0.0000e+00 - val_loss: 5.3152
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.0065 - loss: 5.3148 - val_accuracy: 0.0000e+00 

In [15]:
import glob

# Load test soundscapes
TEST_DIR = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
sample_sub = pd.read_csv('/kaggle/input/competitions/birdclef-2026/sample_submission.csv')

print(f"Submission rows: {len(sample_sub)}")
print(f"Submission columns: {sample_sub.shape[1]}")
print(sample_sub.head())

Submission rows: 3
Submission columns: 235
                                    row_id   1161364    116570   1176823  \
0   BC2026_Test_0001_S05_20250227_010002_5  0.004274  0.004274  0.004274   
1  BC2026_Test_0001_S05_20250227_010002_10  0.004274  0.004274  0.004274   
2  BC2026_Test_0001_S05_20250227_010002_15  0.004274  0.004274  0.004274   

    1491113   1595929    209233     22930     22956     22961  ...   whnjay1  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  ...  0.004274   

     whtdov   whwpic1    y00678    yebcar   yebela1    yecmac    yecpar  \
0  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
1  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   
2  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274  0.004274   

    yehcar1   yeofly1  
0  0.0

In [16]:
def predict_soundscape(audio_path, model, le, duration=5, sr=32000):
    predictions = []
    row_ids = []
    
    filename = os.path.basename(audio_path).replace('.ogg', '')
    y, sr = librosa.load(audio_path, sr=sr)
    total_duration = len(y) / sr
    
    # Split into 5 second segments
    for start in range(0, int(total_duration), duration):
        end = start + duration
        segment = y[start*sr : end*sr]
        
        if len(segment) < duration * sr:
            segment = np.pad(segment, (0, duration*sr - len(segment)))
        
        # Convert to spectrogram
        mel = librosa.feature.melspectrogram(y=segment, sr=sr, n_mels=128)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = tf.image.resize(mel_db[..., np.newaxis], (128,128))
        mel_db = tf.squeeze(mel_db).numpy()
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        
        # Predict
        pred = model.predict(mel_db[np.newaxis, ..., np.newaxis], verbose=0)[0]
        predictions.append(pred)
        row_ids.append(f"{filename}_{end}")
    
    return row_ids, predictions

print("Prediction function ready!")

Prediction function ready!


In [17]:
# Generate predictions on test soundscapes
all_row_ids = []
all_predictions = []

test_files = glob.glob(TEST_DIR + '*.ogg')
print(f"Test files found: {len(test_files)}")

if len(test_files) == 0:
    print("No test files yet — will appear during submission!")
    # Use sample submission as template
    submission = sample_sub.copy()
else:
    for audio_path in test_files:
        row_ids, preds = predict_soundscape(audio_path, model, le)
        all_row_ids.extend(row_ids)
        all_predictions.extend(preds)
    
    # Build submission dataframe
    pred_df = pd.DataFrame(all_predictions, columns=le.classes_)
    pred_df.insert(0, 'row_id', all_row_ids)
    
    # Align with sample submission columns
    submission = sample_sub[['row_id']].merge(pred_df, on='row_id', how='left')
    submission = submission.fillna(1/NUM_CLASSES)

submission.to_csv('submission.csv', index=False)
print("✅ Submission saved!")
print(submission.shape)

Test files found: 0
No test files yet — will appear during submission!
✅ Submission saved!
(3, 235)
